# **Tools in LangGraph / LangChain**

A **Tool** is a Python function that gives an LLM the ability to perform actions outside of its training data. Tools allow the model to access real-time information, databases, APIs, calculations, file systems or any external system.

### What is a Tool exactly?

A tool is usually:

```text
Python Function
       ↓
Wrapped as a Tool
       ↓
Can be called by an LLM
```

The LLM itself does **not execute the function**. It only decides **which tool should be called** and with what arguments. The framework (LangGraph/LangChain) executes the tool and returns the result back to the LLM.

### Why do we need Tools?

LLMs are limited to their training data and cannot:

* Access live information
* Query databases
* Call APIs
* Perform custom business logic
* Read local files

Tools extend the capabilities of the LLM.

### Tool Execution Flow

```text
User Question
      ↓
     LLM
      ↓
Tool Selection
      ↓
Tool Execution
      ↓
Tool Result
      ↓
     LLM
      ↓
 Final Answer
```
### Tool Binding

Tool Binding means attaching available tools to an LLM so that the model knows which tools it can use during execution.

```text
LLM + Tools
      ↓
  Bound LLM
      ↓
Can decide when to call a tool
```

### Key Idea

A tool is typically a Python function wrapped in a format that the LLM can understand and invoke through LangChain/LangGraph. The tool performs the actual work, while the LLM decides **when** and **why** it should be used.


### **@tool Decorator**

The `@tool` decorator converts a normal Python function into a LangChain/LangGraph tool that can be understood and called by an LLM.


from langchain_core.tools import tool
```python
@tool
def add(a: int, b: int) -> int:
    """Adds two numbers"""
    return a + b
```

### What does `@tool` do?

* Registers the function as a tool.
* Exposes the function name and description to the LLM.
* Automatically creates the tool schema from function parameters.
* Allows the LLM to invoke the function when needed.

### Flow

```text
Normal Function
       ↓
     @tool
       ↓
 LangChain Tool
       ↓
 Callable by LLM
```

**Key Idea:** `@tool` is simply a wrapper that converts a Python function into an LLM-usable tool.


### How `@tool` Enhances a Function

Without `@tool`, a function is just normal Python code and only other Python code can call it.

```python
def get_weather(city):
    return f"Weather in {city} is Sunny"
```

After adding `@tool`:

```python
@tool
def get_weather(city):
    return f"Weather in {city} is Sunny"
```

LangChain automatically wraps the function and creates:

* Tool Name → `get_weather`
* Tool Description → From the docstring
* Input Schema → `city: str`
* Output Schema → Function return value

This metadata is exposed to the LLM so it knows:

* What the tool does
* When to use it
* What inputs it requires

```python
# Decorator function
def my_decorator(func):

    def wrapper():
        print("Before function execution")
        
        func()
        
        print("After function execution")

    return wrapper


# Normal function
@my_decorator
def greet():
    print("Hello World")


greet()
```

Output:

Before function execution
Hello World
After function execution

````

### What happens internally?

When Python sees:

```python
@my_decorator
def greet():
    print("Hello World")
````

it actually does:

```python
def greet():
    print("Hello World")

greet = my_decorator(greet)
```

### Flow

```text
greet()
   │
   ▼
wrapper()
   │
   ├──► Before function execution
   │
   ├──► Original greet()
   │        │
   │        ▼
   │    Hello World
   │
   └──► After function execution
```

### Relation with @tool

```python
@tool
def get_weather(city):
    return "Sunny"
```

is conceptually similar to:

```python
get_weather = tool(get_weather)
```

The `tool()` decorator wraps the function and adds metadata so LangChain can expose it to the LLM.


## How a Decorator Works Internally

When Python sees:

```python
@my_decorator
def greet():
    print("Hello World")
```

it automatically converts it into:

```python
def greet():
    print("Hello World")

greet = my_decorator(greet)
```

### What happens next?

1. `greet` is passed to `my_decorator()`.
2. Inside `my_decorator`, `func` now refers to the original `greet` function.
3. A new function called `wrapper()` is created.
4. `wrapper()` contains additional logic before and after calling the original function.
5. `return wrapper` returns the function object (not executing it).return wrapper NOT return wrapper().These are very different.
6. `greet` now points to `wrapper` instead of the original function.

### Important

```python
return wrapper      # Correct
return wrapper()    # Wrong
```

`return wrapper` returns the function itself.

`return wrapper()` executes the function immediately and returns its result.

### Memory View

```text
Before Decoration:
greet ──► Original Function

After Decoration:
greet ──► Wrapper Function ──► Original Function
```

### Execution Flow

```text
greet()
   │
   ▼
wrapper()
   │
   ├──► Before Function Execution
   ├──► Original greet()
   └──► After Function Execution
```

### Flow

```text
greet() ──► wrapper() ──► Before Logic ──► Original greet() ──► After Logic
```

### Key Idea

A decorator replaces the original function reference with an enhanced wrapper function that can execute extra logic before and after the original function.

### Another example to understand decorators
```python
def my_decorator(func):

    def wrapper(a, b):

        print("Before Execution")

        result = func(a, b)   # Call original function

        print("After Execution")

        return result         # Return original result

    return wrapper


@my_decorator
def add(a, b):
    return a + b


print(add(5, 5))

#### What if we don't return?

Suppose:
def wrapper(a, b):

    print("Before")

    result = func(a, b)

    print("After")

    # No return
#Now:
print(add(5,5))
```

Output:
Before
After
None

Why? - Because Python functions return None by default if no explicit return is present.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
import os
load_dotenv()

if os.environ['GOOGLE_API_KEY']:
    print("GEMINI API Key is set.")
else:
    raise ValueError("OpenAI API Key is not set.")

GEMINI API Key is set.


In [12]:
from langchain_core.messages import HumanMessage
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## **DuckDuckGo Search Tool**

The **DuckDuckGo Search Tool** is a web search tool that allows an LLM to retrieve real-time information from the internet. It is commonly used when the model needs up-to-date knowledge that is not available in its training data.

**Key Idea:** DuckDuckGo Tool = Gives the LLM access to live web search results.

In [13]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# @tool converts a normal Python function into an LLM-callable tool
# The LLM can see the tool name, description, inputs and decide when to use it.
@tool
def search_duckduckgo(query: str) -> str:

    # Tool description (docstring)
    # This is exposed to the LLM and helps it understand when to use the tool.
    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""

    # Create an instance of the DuckDuckGo search tool
    # This tool performs real-time web searches.
    duck_search = DuckDuckGoSearchRun()

    # Execute the search using the user's query
    # Example: query = "Latest AI news"
    # The tool sends the query to DuckDuckGo and fetches results.
    return duck_search.invoke(query)


# Flow:
# User Question
#       ↓
#      LLM
#       ↓
# Decides which Tool is Needed
#       ↓
# search_duckduckgo(query)
#       ↓
# DuckDuckGo Search
#       ↓
# Search Results
#       ↓
# Returned to LLM
#       ↓
# Final Response to User

### **Arxiv Query Tool**

In [14]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

# @tool converts this function into an LLM-callable tool.
# The LLM can decide when to use this tool for research-related queries.
@tool
def arxiv_tool(query: str) -> str:

    """This tool allows you to query the arXiv database for research papers."""

    # Create an arXiv API wrapper instance
    # Handles communication with the arXiv database.
    arxiv_query = ArxivQueryRun(
        api_wrapper=ArxivAPIWrapper()
    )

    # Execute the search query on arXiv
    # Example:
    # query = "GraphRAG"
    # query = "Large Language Models"
    return arxiv_query.invoke(query)

###  **Wikipedia Search Tool**

In [15]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

# **Custom Tool**

In [16]:
@tool
def personal_info(name: str):

    """Use this tool to get personal information about Alice, Bob or Charlie. 
    """

    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    # Return information for the given name if it exists in the dictionary;
    # otherwise return the default message "No information available for this person."
    return info.get(name, "No information available for this person.")

## **TOOL BINDING**

### Tool Binding

Tool Binding is the process of attaching one or more tools to an LLM so that the model becomes aware of the available tools and can decide when to invoke them during execution.

**Why do we need it?**

* Makes tools available to the LLM.
* Allows the LLM to choose the appropriate tool.
* Enables access to external data, APIs, and custom functions.
**Key Idea:** Tool binding gives an LLM access to tools, but the LLM decides when and how to use them.

In [17]:
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]

llm_with_tools = llm.bind_tools(tools)

In [18]:
# Send the user query to the LLM that has tools bound to it.
response = llm_with_tools.invoke("What is the latest news on AI?")

# View the tool calls suggested by the LLM.
# This shows which tool the LLM wants to use and the arguments it wants to pass.
response.tool_calls

[{'name': 'search_duckduckgo',
  'args': {'query': 'AI news'},
  'id': '169eb48c-35f8-4584-a9ff-1d1339daa4dd',
  'type': 'tool_call'}]